# Classification de résumés de films — horreur ou comédie ?

Notebook unique, organisé en 8 parties, produisant un modèle entraîné sur
2400 résumés de films (1200 horreur, 1200 comédie) et tous les artefacts
nécessaires à sa mise en production (API, barrière qualité, Docker, Azure).

**Auteur** : Karim — **Date** : 22/10/2026

## Sommaire

1. Audit et nettoyage
2. Découpage du jeu de données
3. Représentation du texte (TF-IDF)
4. Les points de comparaison
5. Recherche des réglages avec KerasTuner
6. Recherche des réglages avec Optuna
7. Verdict sur le test
8. Sauvegarde

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

print("Bibliothèques chargées, graine fixée à", SEED)

Bibliothèques chargées, graine fixée à 42


# Partie 1 — Audit et nettoyage

**Objectif** : charger le CSV, compter les lignes et les films de chaque catégorie,
regarder la longueur des textes, supprimer les espaces en trop et les doublons.

**Pourquoi c'est critique** : un même texte présent dans l'apprentissage **et**
dans le test fausserait le score, car le modèle aurait déjà vu la réponse.
C'est la première règle de tout projet ML.

In [2]:
# Charger le fichier CSV
df = pd.read_csv("textes_films_horreur_comedie.csv")

# Compter les lignes
print("Nombre de lignes :", len(df))

# Afficher les colonnes
print("Colonnes :", list(df.columns))

# Compter les films par catégorie
print("\nRépartition des catégories :")
print(
    df["cible"]
    .value_counts()
    .rename(index={0: "Comédie", 1: "Horreur"})
)

# Vérifier l'équilibre des catégories
print("\nÉquilibre des catégories :")
print(f"Horreur : {(df['cible'] == 1).mean():.1%}")
print(f"Comédie : {(df['cible'] == 0).mean():.1%}")

Nombre de lignes : 2400
Colonnes : ['titre', 'resume', 'cible']

Répartition des catégories :
cible
Comédie    1200
Horreur    1200
Name: count, dtype: int64

Équilibre des catégories :
Horreur : 50.0%
Comédie : 50.0%


**Cellule 2 — Vérifier les données manquantes et les résumés**

In [3]:
# Vérifier les valeurs manquantes
print("Valeurs manquantes :")
print(df.isnull().sum())

# Vérifier les résumés vides
resumes_vides = df["resume"].fillna("").str.strip().eq("")

print("\nNombre de résumés vides :", resumes_vides.sum())

# Calculer la longueur des résumés
longueurs = df["resume"].str.len()

print("\nLongueur des résumés :")
print(longueurs.describe())

Valeurs manquantes :
titre     0
resume    0
cible     0
dtype: int64

Nombre de résumés vides : 0

Longueur des résumés :
count    2400.000000
mean      558.189167
std       349.276409
min       201.000000
25%       335.000000
50%       474.000000
75%       679.000000
max      7104.000000
Name: resume, dtype: float64


**Cellule 3 — Vérifier les doublons**

In [4]:
# Compter les lignes entièrement identiques
print("Lignes entièrement identiques :",
      df.duplicated().sum())

# Compter les résumés identiques
print("Résumés identiques :",
      df["resume"].duplicated().sum())

Lignes entièrement identiques : 54
Résumés identiques : 54


**Cellule 4 — Nettoyer les données**

In [5]:
# Enregistrer le nombre de lignes avant nettoyage 
nb_avant = len(df)

# Nettoyer les espaces au début et à la fin des résumés
df["titre"] = df["titre"].str.strip()
df["resume"] = df["resume"].str.strip()

# Supprimer les lignes entièrement identiques
df = df.drop_duplicates()

# Supprimer les résumés vides 
df = df[df["resume"] != ""]

#Renitialiser les index
df = df.reset_index(drop=True)

# Affichage le bilan
print("Nombre de ligne avant nettoyage :", nb_avant)
print("Nombre de ligne après nettoyage :", len(df))
print("Nombre de lignes supprimées :", nb_avant - len(df))

Nombre de ligne avant nettoyage : 2400
Nombre de ligne après nettoyage : 2346
Nombre de lignes supprimées : 54


**Cellule 5 — La phrase de bilan**

Le jeu de données contient 2 400 lignes au départ, avec 1 200 films d'horreur et 1 200 comédies. Aucune valeur manquante ni aucun résumé vide n'ont été détectés. La longueur moyenne des résumés est d'environ 558 caractères. Après le nettoyage des espaces et la suppression des doublons, il reste 2 346 lignes.

**Découpage du jeu de données**

In [6]:
# X contient les résumés des films
X = df["resume"]

# y  contient les catégories à prédire
y = df["cible"]

print("Nombre de résumés :", len(X))
print("Nombre de catégories :", len(y))

Nombre de résumés : 2346
Nombre de catégories : 2346


**Cellule 2 — Garder 20 % pour la production**

In [7]:
# Garder 20 % des données pour la production
X_reste, X_prod, y_reste, y_prod = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)


print("Données pour la suite :", len(X_reste))
print("Données réservées à la production :", len(X_prod))

Données pour la suite : 1876
Données réservées à la production : 470


**Cellule 3 — Séparer l'entraînement et le test**

In [8]:
# Diviser les données restantes entre entraînement et test
X_train, X_test, y_train, y_test = train_test_split(
    X_reste,
    y_reste,
    test_size=0.25,
    random_state=SEED,
    stratify=y_reste
)

print("Entraînement :", len(X_train))
print("Test :", len(X_test))
print("Production :", len(X_prod))

Entraînement : 1407
Test : 469
Production : 470


**Cellule 4 — Vérifier le découpage**

In [9]:
print("Total après nettoyage :", len(df))
print("Total réparti :", len(X_train) + len(X_test) + len(X_prod))

print("\nRépartition des catégories :")
print("Entraînement :")
print(y_train.value_counts(normalize=True).rename(index={0: "Comédie", 1: "Horreur"}))

print("\nTest :")
print(y_test.value_counts(normalize=True).rename(index={0: "Comédie", 1: "Horreur"}))

print("\nProduction :")
print(y_prod.value_counts(normalize=True).rename(index={0: "Comédie", 1: "Horreur"}))

Total après nettoyage : 2346
Total réparti : 2346

Répartition des catégories :
Entraînement :
cible
Comédie    0.509595
Horreur    0.490405
Name: proportion, dtype: float64

Test :
cible
Comédie    0.509595
Horreur    0.490405
Name: proportion, dtype: float64

Production :
cible
Comédie    0.508511
Horreur    0.491489
Name: proportion, dtype: float64


**Représentation du texte (TF-IDF)**

In [10]:
# Importer l'outil TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

**Créer et entraîner TF-IDF**

In [11]:
# Créer le vectoriseur TF-IDF
vectoriseur = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

# Apprendre le vocabulaire uniquement sur les données d'entraînement
X_train_tfidf = vectoriseur.fit_transform(X_train)

# Transformer les données de test et de production
X_test_tfidf = vectoriseur.transform(X_test)
X_prod_tfidf = vectoriseur.transform(X_prod)

print("Taille des données d'entrainement :", X_train_tfidf.shape)
print("Taille des données de test :", X_test_tfidf.shape)
print("Taille des données de production :", X_prod_tfidf.shape)

Taille des données d'entrainement : (1407, 5000)
Taille des données de test : (469, 5000)
Taille des données de production : (470, 5000)


In [12]:
#Vérifier le résultat

# Afficher quelques mots appris par TF-IDF
print("Exemples de mots du vocabulaire :")
print(vectoriseur.get_feature_names_out()[:20])

# On vérifie le nombre de caractéristiques
print("\nNombre de caractéristique :",  len((vectoriseur.get_feature_names_out())))

# vérifier les dimensions
controle(
    X_train_tfidf.shape[0] == len(y_train),
    "Le nombre de lignes correspond aux catégories d'entrainement",
    "Le nombre de lignes ne correspond pas aux catégories d'entrainement "
) 


Exemples de mots du vocabulaire :
['000' '10' '100' '11' '12' '13' '13th' '14' '15' '18' '180sm' '18th'
 '1950s' '1957' '1970s' '1972' '1974' '1980s' '1987' '1988']

Nombre de caractéristique : 5000
✅ Le nombre de lignes correspond aux catégories d'entrainement


**Les points de comparaison**

# Partie 4 — Les points de comparaison

**Objectif** : entraîner 3 modèles simples et comparer leur **F1-score** sur la validation.
Le jeu de test sera conservé pour l'évaluation finale.

**Modèles comparés** :
1. **Régression logistique** — baseline linéaire (rapide, interprétable)
2. **Bayes naïf multinomial** — baseline probabiliste, très adapté au TF-IDF
3. **Réseau très simple** — 1 couche cachée, un peu de dropout, **sans réglage**

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

**Tester la régression logistique**

In [14]:
# Séparer une partie des données d'entraînement pour la validation
X_train_part, X_val,y_train_part, y_val = train_test_split(
    X_train_tfidf,
    y_train,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train
)

# crée et entrainer  le modèle
model_logistique = LogisticRegression(max_iter=1000)
model_logistique.fit(X_train_part,y_train_part)

#predire les catégories de validation
pred_logistique = model_logistique.predict(X_val)

# Calculer le F1-score
f1_logistique = f1_score(y_val, pred_logistique)

print("F1-score de la régression logistique :", round(f1_logistique, 3))

F1-score de la régression logistique : 0.845


**Tester Naive Bayes**

In [15]:
# Créer le modèle
modele_nb = MultinomialNB()

# Entrainer le modèle
modele_nb.fit(X_train_part, y_train_part)

#Faire les prédictions
pred_nb = modele_nb.predict(X_val)

# Calculer le F1-score
f1_nb = f1_score(y_val, pred_nb)

print("F1-score de Naive Bayes :", round(f1_nb, 3))

F1-score de Naive Bayes : 0.864


**Créer un réseau de neurones simple**

In [16]:
def creer_model_simple(nb_mots):
    modele = keras.Sequential([
        layers.Input(shape=(nb_mots,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1,activation="sigmoid")
    ])
    modele.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return modele

**Entraîner et évaluer le réseau**

In [17]:
# Créer le réseau
modele_reseau = creer_model_simple(X_train_tfidf.shape[1])

# Entrainer le réseau
historique = modele_reseau.fit(
    X_train_part.toarray(),
    y_train_part,
    validation_data=(X_val.toarray(), y_val),
    epochs=10,
    batch_size=32,
    verbose=0
)

# Prédire les catégories de validation
probas = modele_reseau.predict(X_val.toarray(), verbose=0).ravel()
pred_reseau = (probas >= 0.5).astype(int)

# Calculer le F1-score
f1_reseau = f1_score(y_val,pred_reseau)

print("F1-score du réseau de neurones :", round(f1_reseau,3))

F1-score du réseau de neurones : 0.858


**Comparer les trois modèles**

In [18]:
resultats_simples = pd.DataFrame({
    "Modèle": [
        "Régression logistique",
        "Naive Bayes",
        "Réseau de neurones"
    ],
    "F1-score": [
        f1_logistique,
        f1_nb,
        f1_reseau
    ]
})

print(resultats_simples.sort_values("F1-score", ascending=False))

                  Modèle  F1-score
1            Naive Bayes  0.864469
2     Réseau de neurones  0.858238
0  Régression logistique  0.844961


**Tester avec 5 graines**

In [19]:
graines = [11, 22, 33, 44, 55]
resultats_graines = []

for graine in graines:

    # Créer une nouvelle séparation entraînement / validation
    X_train_part, X_val, y_train_part, y_val = train_test_split(
        X_train_tfidf,
        y_train,
        test_size=0.20,
        random_state=graine,
        stratify=y_train
    )

    # Régression logistique
    modele_logistique = LogisticRegression(max_iter=1000)
    modele_logistique.fit(X_train_part, y_train_part)
    pred_log = modele_logistique.predict(X_val)

    resultats_graines.append({
        "Graine": graine,
        "Modèle": "Régression logistique",
        "F1-score": f1_score(y_val, pred_log)
    })

    # Naive Bayes
    modele_nb = MultinomialNB()
    modele_nb.fit(X_train_part, y_train_part)
    pred_nb = modele_nb.predict(X_val)

    resultats_graines.append({
        "Graine": graine,
        "Modèle": "Naive Bayes",
        "F1-score": f1_score(y_val, pred_nb)
    })

resultats_5_graines = pd.DataFrame(resultats_graines)
print(resultats_5_graines)

   Graine                 Modèle  F1-score
0      11  Régression logistique  0.881481
1      11            Naive Bayes  0.884892
2      22  Régression logistique  0.853846
3      22            Naive Bayes  0.884058
4      33  Régression logistique  0.837736
5      33            Naive Bayes  0.848057
6      44  Régression logistique  0.871212
7      44            Naive Bayes  0.878571
8      55  Régression logistique  0.798419
9      55            Naive Bayes  0.847584


**Tester le réseau de neurones avec 5 graines**

In [20]:
resultats_reseau = []

graines = [11, 22, 33, 44, 55]

for graine in graines:

    # Séparer les données d'entraînement et de validation
    X_train_part, X_val, y_train_part, y_val = train_test_split(
        X_train_tfidf,
        y_train,
        test_size=0.20,
        random_state=graine,
        stratify=y_train
    )

    # Fixer la graine pour rendre l'entraînement reproductible
    keras.utils.set_random_seed(graine)

    # Créer le réseau de neurones
    modele = creer_model_simple(X_train_tfidf.shape[1])

    # Entraîner le réseau
    modele.fit(
        X_train_part.toarray(),
        y_train_part,
        validation_data=(X_val.toarray(), y_val),
        epochs=10,
        batch_size=32,
        verbose=0
    )

    # Prédire les catégories
    probabilites = modele.predict(X_val.toarray(), verbose=0).ravel()
    predictions = (probabilites >= 0.5).astype(int)

    # Calculer le F1-score
    score = f1_score(y_val, predictions)

    resultats_reseau.append({
        "Graine": graine,
        "Modèle": "Réseau de neurones",
        "F1-score": score
    })

# Afficher les résultats
resultats_reseau = pd.DataFrame(resultats_reseau)
print(resultats_reseau.round(3))

   Graine              Modèle  F1-score
0      11  Réseau de neurones     0.860
1      22  Réseau de neurones     0.871
2      33  Réseau de neurones     0.847
3      44  Réseau de neurones     0.862
4      55  Réseau de neurones     0.805


In [21]:
# Réunir les résultats des trois modèles
resultats_complets = pd.concat(
    [resultats_5_graines, resultats_reseau],
    ignore_index=True
)

# Afficher tous les scores
print(resultats_complets.round(3))

    Graine                 Modèle  F1-score
0       11  Régression logistique     0.881
1       11            Naive Bayes     0.885
2       22  Régression logistique     0.854
3       22            Naive Bayes     0.884
4       33  Régression logistique     0.838
5       33            Naive Bayes     0.848
6       44  Régression logistique     0.871
7       44            Naive Bayes     0.879
8       55  Régression logistique     0.798
9       55            Naive Bayes     0.848
10      11     Réseau de neurones     0.860
11      22     Réseau de neurones     0.871
12      33     Réseau de neurones     0.847
13      44     Réseau de neurones     0.862
14      55     Réseau de neurones     0.805


In [22]:
# Calculer les statistiques pour chaque modèle
synthese = (
    resultats_complets
    .groupby("Modèle")["F1-score"]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

# Calculer la marge de bruit indicative
n_validation = int(len(y_val))

synthese["Marge de bruit"] = synthese["mean"].apply(
    lambda p: 1.96 * np.sqrt(p * (1 - p) / n_validation)
)

print(synthese.round(3))

                  Modèle   mean    std    min    max  Marge de bruit
0            Naive Bayes  0.869  0.019  0.848  0.885           0.039
1  Régression logistique  0.849  0.033  0.798  0.881           0.042
2     Réseau de neurones  0.849  0.026  0.805  0.871           0.042


Après avoir testé les trois modèles avec cinq graines différentes, Naive Bayes obtient le meilleur F1-score moyen avec 0,869. Le réseau de neurones obtient 0,853 et la régression logistique 0,849. Naive Bayes semble donc être le meilleur modèle de base pour notre classification. Dans les prochaines parties, nous allons essayer d'améliorer les résultats du réseau de neurones en modifiant ses paramètres.

# Partie 5 — Recherche des réglages avec KerasTuner

**Objectif :** tester plusieurs configurations du réseau de neurones pour trouver celle qui obtient le meilleur F1-score sur la validation.

Le jeu de test reste de côté pour l'évaluation finale.

In [23]:
import keras_tuner as kt
from tensorflow.keras import regularizers

In [24]:
def construire_modele(hp):
    """Construit un réseau à partir de 5 hyperparamètres choisis par KerasTuner."""
    modele = keras.Sequential()
    modele.add(layers.Input(shape=(X_train_part.shape[1],)))

    # Nombre de couches (1 à 3), chacune avec neurones / dropout / L2
    for i in range(hp.Int("nb_couches", 1, 3)):
        modele.add(layers.Dense(
            units=hp.Int(f"neurones_{i}", min_value=16, max_value=128, step=16),
            activation="relu",
            kernel_regularizer=regularizers.l2(
                hp.Choice("l2", [0.0, 1e-4, 1e-3, 1e-2])
            ),
        ))
        modele.add(layers.Dropout(
            hp.Choice("dropout", [0.0, 0.1, 0.2, 0.3, 0.5])
        ))

    modele.add(layers.Dense(1, activation="sigmoid"))

    modele.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=hp.Choice("lr", [1e-4, 1e-3, 1e-2])
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return modele

Le callback F1

In [25]:
class F1Callback(keras.callbacks.Callback):
    """Calcule le F1 sur validation à chaque epoch et l'expose dans logs."""
    def __init__(self, X_val, y_val):
        self.X_val = X_val
        self.y_val = y_val

    def on_epoch_end(self, epoch, logs=None):
        proba = self.model.predict(self.X_val, verbose=0).ravel()
        pred = (proba >= 0.5).astype(int)
        f1 = f1_score(self.y_val, pred)
        if logs is not None:
            logs["val_f1"] = f1

 Lancement du tuner

In [26]:
# Conversion en dense (Keras préfère, plus rapide avec KerasTuner)
X_train_dense = X_train_part.toarray()
X_val_dense = X_val.toarray()

tuner = kt.RandomSearch(
    construire_modele,
    objective=kt.Objective("val_f1", direction="max"),
    max_trials=20,
    executions_per_trial=1,
    overwrite=True,
    directory="kt_films",
    project_name="horreur_vs_comedie",
    seed=SEED,
)

tuner.search(
    X_train_dense,
    y_train_part,
    validation_data=(X_val_dense, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[F1Callback(X_val_dense, y_val)],
    verbose=0,
)

print("Recherche KerasTuner terminée.")


Recherche KerasTuner terminée.


**Résultats**

In [27]:
meilleurs_essais = tuner.oracle.get_best_trials(num_trials=5)

lignes = []
for t in meilleurs_essais:
    hps = t.hyperparameters.values
    lignes.append({
        "essai":       t.trial_id,
        "F1 (val)":    round(t.score, 4),
        "nb_couches":  hps.get("nb_couches"),
        "neurones_c1": hps.get("neurones_0"),
        "dropout":     hps.get("dropout"),
        "l2":          hps.get("l2"),
        "lr":          hps.get("lr"),
    })

tableau_tuner = pd.DataFrame(lignes)
print("=== Top 5 KerasTuner ===")
print(tableau_tuner.to_string(index=False))

meilleur_f1_kt = tableau_tuner["F1 (val)"].max()
print(f"\nMeilleur F1 KerasTuner : {meilleur_f1_kt:.4f}")
print("Meilleurs hyperparamètres :")
for k, v in meilleurs_essais[0].hyperparameters.values.items():
    print(f"   {k} = {v}")

=== Top 5 KerasTuner ===
essai  F1 (val)  nb_couches  neurones_c1  dropout     l2     lr
   01    0.8502           2           80      0.2 0.0000 0.0001
   08    0.8496           1           16      0.5 0.0001 0.0010
   05    0.8467           1           16      0.5 0.0001 0.0010
   04    0.8464           2           16      0.3 0.0000 0.0100
   14    0.8454           2           64      0.3 0.0001 0.0010

Meilleur F1 KerasTuner : 0.8502
Meilleurs hyperparamètres :
   nb_couches = 2
   neurones_0 = 80
   l2 = 0.0
   dropout = 0.2
   lr = 0.0001
   neurones_1 = 128


**Observations KerasTuner** :

- **Meilleur F1 (validation)** : 0,8502
- **Hyperparamètres retenus** : 2 couches, 80 + 128 neurones, dropout 0.2, L2 0, lr 1e-4
- **Référence à battre (Naive Bayes)** : 0,869
- **Seuil qualité (0,85)** : atteint de justesse (+0,0002)

**Interprétation** : KerasTuner trouve un réseau à 2 couches assez large,
modérément régularisé. Le score reste proche de Naive Bayes, et l'écart
sera jugé en partie 7 avec la marge de bruit.

**Recherche des réglages avec Optuna**

In [28]:
import optuna
print("Optuna est prêt !")

Optuna est prêt !


**La fonction d'objectif Optuna**

In [29]:

def objectif_optuna(trial):
    """Fonction objectif : construit et entraîne un réseau selon les hyperparamètres
    proposés par Optuna, puis renvoie le F1 de validation (à maximiser)."""
    keras.utils.set_random_seed(SEED)

    modele = keras.Sequential()
    modele.add(layers.Input(shape=(X_train_dense.shape[1],)))

    nb_couches = trial.suggest_int("nb_couches", 1, 3)
    for i in range(nb_couches):
        modele.add(layers.Dense(
            units=trial.suggest_int(f"neurones_{i}", 16, 128, step=16),
            activation="relu",
            kernel_regularizer=regularizers.l2(
                trial.suggest_categorical("l2", [0.0, 1e-4, 1e-3, 1e-2])
            ),
        ))
        modele.add(layers.Dropout(
            trial.suggest_categorical("dropout", [0.0, 0.1, 0.2, 0.3, 0.5])
        ))

    modele.add(layers.Dense(1, activation="sigmoid"))

    modele.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=trial.suggest_categorical("lr", [1e-4, 1e-3, 1e-2])
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    modele.fit(
        X_train_dense, y_train_part,
        validation_data=(X_val_dense, y_val),
        epochs=20, batch_size=32,
        callbacks=[F1Callback(X_val_dense, y_val)],
        verbose=0,
    )

    # On récupère le F1 final après entraînement
    proba = modele.predict(X_val_dense, verbose=0).ravel()
    pred = (proba >= 0.5).astype(int)
    return f1_score(y_val, pred)

**Lancer Optuna**

In [30]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

etude = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

etude.optimize(
    objectif_optuna,
    n_trials=20,
    show_progress_bar=True
)

print("\nRecherche Optuna terminée.")

  0%|          | 0/20 [00:00<?, ?it/s]


Recherche Optuna terminée.


**Résultats**

In [31]:
# Top 5 des essais
top5_optuna = sorted(etude.trials, key=lambda t: t.value, reverse=True)[:5]

lignes = []
for t in top5_optuna:
    p = t.params
    lignes.append({
        "essai":       t.number,
        "F1 (val)":    round(t.value, 4) if t.value is not None else None,
        "nb_couches":  p.get("nb_couches"),
        "neurones_c1": p.get("neurones_0"),
        "dropout":     p.get("dropout"),
        "l2":          p.get("l2"),
        "lr":          p.get("lr"),
    })

tableau_optuna = pd.DataFrame(lignes)
print("=== Top 5 Optuna ===")
print(tableau_optuna.to_string(index=False))

meilleur_f1_optuna = etude.best_value
print(f"\nMeilleur F1 Optuna : {meilleur_f1_optuna:.4f}")
print("Meilleurs hyperparamètres :")
for k, v in etude.best_params.items():
    print(f"   {k} = {v}")

=== Top 5 Optuna ===
 essai  F1 (val)  nb_couches  neurones_c1  dropout     l2     lr
    11    0.8327           2          112      0.2 0.0000 0.0001
    12    0.8249           2           80      0.2 0.0000 0.0001
     7    0.8226           3           32      0.2 0.0001 0.0010
     0    0.8175           2          128      0.1 0.0000 0.0001
    10    0.8140           2           48      0.2 0.0001 0.0010

Meilleur F1 Optuna : 0.8327
Meilleurs hyperparamètres :
   nb_couches = 2
   neurones_0 = 112
   l2 = 0.0
   dropout = 0.2
   neurones_1 = 128
   lr = 0.0001


**Comparaison KerasTuner vs Optuna**

In [32]:
comparaison_outils = pd.DataFrame({
    "Outil":        ["KerasTuner", "Optuna"],
    "Meilleur F1":  [meilleur_f1_kt, meilleur_f1_optuna],
    "Écart vs NB":  [meilleur_f1_kt - 0.869, meilleur_f1_optuna - 0.869],
    "Écart vs seuil (0.85)": [meilleur_f1_kt - 0.85, meilleur_f1_optuna - 0.85],
}).round(4)

print(comparaison_outils.to_string(index=False))

# Marge de bruit
n_val = len(y_val)
p_moy = (meilleur_f1_kt + meilleur_f1_optuna) / 2
marge_bruit = 1.96 * np.sqrt(p_moy * (1 - p_moy) / n_val)

print(f"\nMarge de bruit (n={n_val}) : ± {marge_bruit:.4f}")
ecart = abs(meilleur_f1_kt - meilleur_f1_optuna)
print(f"Écart KerasTuner vs Optuna : {ecart:.4f}")
print(f"→ L'écart est {'DANS' if ecart < marge_bruit else 'AU-DELÀ DE'} la marge de bruit")

     Outil  Meilleur F1  Écart vs NB  Écart vs seuil (0.85)
KerasTuner       0.8502      -0.0188                 0.0002
    Optuna       0.8327      -0.0363                -0.0173

Marge de bruit (n=282) : ± 0.0426
Écart KerasTuner vs Optuna : 0.0175
→ L'écart est DANS la marge de bruit


** Sauvegarde des meilleurs params**

In [33]:
# Stockage pour la partie 7
meilleurs_params_optuna = etude.best_params.copy()

print("Params Optuna sauvegardés :")
for k, v in meilleurs_params_optuna.items():
    print(f"   {k} = {v}")

Params Optuna sauvegardés :
   nb_couches = 2
   neurones_0 = 112
   l2 = 0.0
   dropout = 0.2
   neurones_1 = 128
   lr = 0.0001


# Partie 7 — Verdict sur le test

**Objectif** : on prend les meilleurs paramètres trouvés (KerasTuner, 0.8502),
on réentraîne sur **tout** l'apprentissage, sur **5 graines**, et on mesure
sur le test **une seule fois**.

**Modèles comparés** :
1. Réseau optimisé (meilleurs params KerasTuner)
2. Réseau de départ (partie 4, sans réglage)
3. Naive Bayes (référence : 0.869 en validation)

**Question** : le réseau est-il vraiment meilleur, ou l'écart est-il dans la marge de bruit ?

**Reconstruire un réseau depuis un dictionnaire**

In [34]:
def construire_depuis_params(params):
    """Reconstruit un réseau depuis un dict d'hyperparamètres."""
    modele = keras.Sequential()
    modele.add(layers.Input(shape=(X_train_tfidf.shape[1],)))

    for i in range(params["nb_couches"]):
        modele.add(layers.Dense(
            units=params[f"neurones_{i}"],
            activation="relu",
            kernel_regularizer=regularizers.l2(params["l2"])
        ))
        modele.add(layers.Dropout(params["dropout"]))

    modele.add(layers.Dense(1, activation="sigmoid"))
    modele.compile(
        optimizer=keras.optimizers.Adam(learning_rate=params["lr"]),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return modele

**Les 3 modèles sur le test**

In [35]:
# Les meilleurs params trouvés par KerasTuner
meilleurs_params = {
    "nb_couches": 2,
    "neurones_0": 80,
    "neurones_1": 128,
    "dropout":    0.2,
    "l2":         0.0,
    "lr":         1e-4,
}

# Conversion en dense une seule fois
X_train_full = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

graines = [11, 22, 33, 44, 55]
scores_opt = []
scores_depart = []

for graine in graines:
    # --- Réseau optimisé ---
    keras.utils.set_random_seed(graine)
    m = construire_depuis_params(meilleurs_params)
    m.fit(X_train_full, y_train, epochs=20, batch_size=32, verbose=0)
    pred = (m.predict(X_test_dense, verbose=0).ravel() >= 0.5).astype(int)
    scores_opt.append(f1_score(y_test, pred))

    # --- Réseau de départ (partie 4) ---
    keras.utils.set_random_seed(graine)
    m = creer_model_simple(X_train_tfidf.shape[1])
    m.fit(X_train_full, y_train, epochs=20, batch_size=32, verbose=0)
    pred = (m.predict(X_test_dense, verbose=0).ravel() >= 0.5).astype(int)
    scores_depart.append(f1_score(y_test, pred))

# --- Naive Bayes (déterministe, une seule fois) ---
nb_final = MultinomialNB()
nb_final.fit(X_train_tfidf, y_train)
f1_nb_test = f1_score(y_test, nb_final.predict(X_test_tfidf))

# Affichage
print(f"Réseau optimisé  : {np.mean(scores_opt):.4f} ± {np.std(scores_opt):.4f}")
print(f"Réseau de départ : {np.mean(scores_depart):.4f} ± {np.std(scores_depart):.4f}")
print(f"Naive Bayes      : {f1_nb_test:.4f}")

Réseau optimisé  : 0.8621 ± 0.0046
Réseau de départ : 0.8512 ± 0.0031
Naive Bayes      : 0.8750


**Tableau + écart + marge de bruit**

In [36]:
# Tableau récapitulatif
verdict = pd.DataFrame({
    "Modèle": ["Réseau optimisé", "Réseau de départ", "Naive Bayes"],
    "F1 test": [
        np.mean(scores_opt),
        np.mean(scores_depart),
        f1_nb_test
    ],
    "Écart-type": [
        np.std(scores_opt),
        np.std(scores_depart),
        0.0
    ]
}).round(4)

print("=== Verdict sur le test ===")
print(verdict.to_string(index=False))

# Marge de bruit sur le test
n_test = len(y_test)
p = np.mean(scores_opt)
marge = 1.96 * np.sqrt(p * (1 - p) / n_test)

print(f"\nMarge de bruit sur le test (n={n_test}) : ± {marge:.4f}")

# Comparaison réseau vs Naive Bayes
ecart = np.mean(scores_opt) - f1_nb_test
print(f"Écart réseau optimisé vs Naive Bayes : {ecart:+.4f}")

if abs(ecart) < marge:
    print("→ L'écart est DANS la marge de bruit : aucune différence démontrée")
elif ecart > 0:
    print("→ Le réseau est MEILLEUR que Naive Bayes")
else:
    print("→ Naive Bayes reste MEILLEUR que le réseau")

=== Verdict sur le test ===
          Modèle  F1 test  Écart-type
 Réseau optimisé   0.8621      0.0046
Réseau de départ   0.8512      0.0031
     Naive Bayes   0.8750      0.0000

Marge de bruit sur le test (n=469) : ± 0.0312
Écart réseau optimisé vs Naive Bayes : -0.0129
→ L'écart est DANS la marge de bruit : aucune différence démontrée


### Verdict sur les résultats

- **Réseau optimisé :** F1 = 0,8621 ± 0,0046.
- **Réseau de départ :** F1 = 0,8512 ± 0,0031.
- **Naive Bayes :** F1 = 0,8750.

### Conclusions

- L'optimisation améliore le F1 du réseau de **1,09 point** par rapport au modèle de départ.
- Naive Bayes obtient le meilleur score brut, mais l'écart avec le réseau optimisé (1,29 point) reste inférieur à la marge d'incertitude estimée (±3,12 points). On ne peut donc pas affirmer qu'il est significativement meilleur.
- Les trois modèles atteignent le seuil de qualité fixé à **0,85**.
- Les résultats restent à interpréter avec prudence, car le jeu de test contient seulement 469 exemples et la validation 282 lignes.

### Choix pour la production

Je retiens le **réseau optimisé**, car il améliore les résultats du réseau de départ et respecte le seuil de qualité demandé. De plus, le brief demande un fichier `model.keras`. Le choix reste cohérent, même si Naive Bayes obtient un meilleur F1 brut.

# Partie 8 — Sauvegarde

**Objectif** : créer un dossier `model/` qui contient **tout ce qu'il faut**
pour réutiliser le modèle ailleurs (API, Docker, Azure) :

- `model.keras` : le réseau entraîné (meilleurs params)
- `vectoriseur.joblib` : le TF-IDF (indispensable pour transformer le texte)
- `metrics.json` : version, F1 test, taille du test, versions des bibliothèques
- `reference.csv` : un petit échantillon de données de référence

Le dossier `model/` sera copié dans l'image Docker pour que l'API puisse
charger le modèle au démarrage.

In [37]:
import os
import json
import joblib
import sklearn
import tensorflow as tf

# Créer le dossier model/ (s'il existe déjà, on le garde)
os.makedirs("model", exist_ok=True)

# Réentraîner le modèle final sur tout l'apprentissage avec la graine SEED
keras.utils.set_random_seed(SEED)
modele_final = construire_depuis_params(meilleurs_params)
modele_final.fit(
    X_train_full, y_train,
    epochs=20, batch_size=32,
    verbose=0
)

print("Modèle final entraîné sur", X_train_full.shape[0], "lignes d'apprentissage")

Modèle final entraîné sur 1407 lignes d'apprentissage


**Sauvegarder le réseau et le TF-IDF**

In [38]:
# 1) Le réseau au format .keras
chemin_modele = "model/model.keras"
modele_final.save(chemin_modele)
print(f"Modèle sauvegardé : {chemin_modele} "
      f"({os.path.getsize(chemin_modele) / 1024:.1f} Ko)")

# 2) Le TF-IDF avec joblib
chemin_vectoriseur = "model/vectoriseur.joblib"
joblib.dump(vectoriseur, chemin_vectoriseur)
print(f"Vectoriseur sauvegardé : {chemin_vectoriseur} "
      f"({os.path.getsize(chemin_vectoriseur) / 1024:.1f} Ko)")

Modèle sauvegardé : model/model.keras (4840.9 Ko)
Vectoriseur sauvegardé : model/vectoriseur.joblib (179.1 Ko)


**Créer metrics.json**

In [39]:
# Calculer le F1 final sur le test
pred_final = (modele_final.predict(X_test_dense, verbose=0).ravel() >= 0.5).astype(int)
f1_final = f1_score(y_test, pred_final)

# Construire le dictionnaire de métriques
metrics = {
    "version": "1.0.0",
    "model_type": "keras_binary_classifier",
    "f1_test": round(float(f1_final), 4),
    "test_size": int(len(y_test)),
    "n_train": int(len(y_train)),
    "n_features": int(X_train_tfidf.shape[1]),
    "seuil_qualite": 0.85,
    "passe_barriere": bool(f1_final >= 0.85),
    "hyperparametres": meilleurs_params,
    "versions": {
        "tensorflow": tf.__version__,
        "scikit_learn": sklearn.__version__,
        "python": f"{__import__('sys').version_info.major}.{__import__('sys').version_info.minor}"
    }
}

# Écrire en UTF-8 (obligatoire pour éviter l'erreur sur Windows)
with open("model/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("metrics.json créé :")
print(json.dumps(metrics, indent=2, ensure_ascii=False))

metrics.json créé :
{
  "version": "1.0.0",
  "model_type": "keras_binary_classifier",
  "f1_test": 0.8571,
  "test_size": 469,
  "n_train": 1407,
  "n_features": 5000,
  "seuil_qualite": 0.85,
  "passe_barriere": true,
  "hyperparametres": {
    "nb_couches": 2,
    "neurones_0": 80,
    "neurones_1": 128,
    "dropout": 0.2,
    "l2": 0.0,
    "lr": 0.0001
  },
  "versions": {
    "tensorflow": "2.22.0-rc0",
    "scikit_learn": "1.9.1",
    "python": "3.14"
  }
}


**Créer reference.csv**

In [40]:
# Échantillon de référence : 20 lignes du jeu d'apprentissage
reference = df.sample(20, random_state=SEED)[["titre", "resume", "cible"]]
reference.to_csv("model/reference.csv", index=False, encoding="utf-8")
print(f"reference.csv créé : {len(reference)} lignes")

reference.csv créé : 20 lignes


**Vérifier le dossier model/**

In [41]:
# Vérifier que les 4 fichiers sont là
attendus = ["model.keras", "vectoriseur.joblib", "metrics.json", "reference.csv"]
contenus = os.listdir("model")

print("Contenu du dossier model/ :")
for f in attendus:
    present = "✅" if f in contenus else "❌"
    taille = os.path.getsize(f"model/{f}") if f in contenus else 0
    print(f"  {present} {f:25} ({taille / 1024:.1f} Ko)")

# Contrôle final
controle(
    all(f in contenus for f in attendus),
    "Les 4 fichiers sont présents dans model/",
    "Il manque un fichier dans model/"
)

Contenu du dossier model/ :
  ✅ model.keras               (4840.9 Ko)
  ✅ vectoriseur.joblib        (179.1 Ko)
  ✅ metrics.json              (0.5 Ko)
  ✅ reference.csv             (10.7 Ko)
✅ Les 4 fichiers sont présents dans model/


**Test de rechargement**

In [42]:
# Recharger tout et refaire une prédiction
modele_recharge = keras.models.load_model("model/model.keras")
vec_recharge = joblib.load("model/vectoriseur.joblib")

# Prendre un texte quelconque du test
texte_test = df["resume"].iloc[0]
print(f"Texte test : {texte_test[:80]}...")

# Prédire
vecteur = vec_recharge.transform([texte_test])
proba = float(modele_recharge.predict(vecteur.toarray(), verbose=0).ravel()[0])
classe = "horreur" if proba >= 0.5 else "comédie"
print(f"Prédiction : {classe} (proba horreur = {proba:.3f})")

# Contrôle
controle(
    0 <= proba <= 1,
    "Rechargement OK : le modèle prédit bien après sauvegarde/rechargement",
    "Problème de rechargement : vérifiez les fichiers"
)

Texte test : Roberto has a new job as the teacher at a nursery school. The first child he mee...
Prédiction : comédie (proba horreur = 0.404)
✅ Rechargement OK : le modèle prédit bien après sauvegarde/rechargement
